# 07 - System Integration
## Bringing All Four Modules Together: NewsBotIntelligenceSystem 2.0

Integrates classification, topic modeling, summarization, multilingual analysis, and the conversational interface into a single end-to-end pipeline class, and generates the final business insight report.> **Setup note:** This notebook uses the `src/` package from the project root.
> If running in Google Colab, first mount/clone the repo so `src/` is on the path,
> and place the Kaggle `BBC News Train.csv` in `data/raw/` for the real dataset
> (otherwise the bundled offline sample in `data/sample/` is used automatically).


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

from src.data_processing.data_loader import load_news_data, dataset_source
print("Dataset source:", dataset_source())


Dataset source: Bundled sample dataset (demo only): /home/claude/ITAI2373-NewsBot-Final/data/sample/sample_news.csv


In [2]:
from src.data_processing.text_preprocessor import TextPreprocessor
from src.data_processing.feature_extractor import FeatureExtractor
from src.analysis.classifier import NewsClassifier
from src.analysis.sentiment_analyzer import SentimentAnalyzer
from src.analysis.ner_extractor import NERExtractor
from src.analysis.topic_modeler import TopicModeler
from src.language_models.summarizer import Summarizer
from src.language_models.generator import InsightGenerator
from src.multilingual.language_detector import LanguageDetector
from src.conversation.query_processor import QueryProcessor
from src.conversation.response_generator import ResponseGenerator
from src.utils.export import build_summary_report

## `NewsBotIntelligenceSystem`: the unified pipeline class
This mirrors (and extends, with topics + multilingual + conversation) the `NewsBotIntelligenceSystem` class built in the midterm project.

In [3]:
class NewsBotIntelligenceSystem:
    """Complete NewsBot 2.0 pipeline: classification, sentiment, NER, topics,
    summarization, multilingual detection, and conversational querying."""

    def __init__(self, n_topics=6):
        self.preprocessor = TextPreprocessor()
        self.feature_extractor = FeatureExtractor(max_features=5000)
        self.classifier = NewsClassifier()
        self.sentiment_analyzer = SentimentAnalyzer()
        self.ner_extractor = NERExtractor()
        self.topic_modeler = TopicModeler(n_topics=n_topics, method="lda")
        self.summarizer = Summarizer()
        self.generator = InsightGenerator()
        self.language_detector = LanguageDetector()
        self.query_processor = None
        self.df = None

    def train(self, df):
        df = self.preprocessor.preprocess_dataframe(df, text_col="content", title_col="title")
        df = self.sentiment_analyzer.analyze_dataframe(df, text_col="content")
        X = self.feature_extractor.fit_transform(df["content_processed"])
        self.classifier_results = self.classifier.train(X, df["category"])
        self.topic_modeler.fit_transform(df["content_processed"])
        df["topic_id"] = self.topic_modeler.dominant_topic_per_doc()
        self.df = df
        self.query_processor = QueryProcessor(df, text_col="content")
        return self

    def analyze(self, title, content):
        clean = self.preprocessor.preprocess_text(content)
        X = self.feature_extractor.transform([clean])
        pred = self.classifier.predict_with_confidence(X)[0]
        sentiment = self.sentiment_analyzer.analyze(content)
        entities = self.ner_extractor.extract_entities(content)
        summary = self.summarizer.summarize(content, n_sentences=2)
        lang = self.language_detector.detect(content)
        article = {
            "title": title, "category": pred["label"], "confidence": pred["confidence"],
            "sentiment_label": sentiment["label"], "entities": entities,
            "summary": summary, "language": lang["language_name"],
        }
        return self.generator.enhance_article(article)

    def ask(self, query):
        result = self.query_processor.process(query)
        return ResponseGenerator().format_response(result)

    def business_report(self):
        summary = build_summary_report(self.df, self.classifier_results)
        return {"summary": summary, "insights": self.generator.generate_business_insights(summary)}


## Train the integrated system

In [4]:
system = NewsBotIntelligenceSystem(n_topics=6)
df = load_news_data()
system.train(df)
print("Training complete. Best classifier:", system.classifier.best_model_name)

Training complete. Best classifier: Naive Bayes


## End-to-end test on new articles

In [5]:
test_articles = [
    {"title": "Microsoft Acquires AI Startup for $2 Billion",
     "content": "Microsoft Corporation announced today the acquisition of an artificial intelligence "
                "startup for $2 billion. CEO Satya Nadella said the deal will strengthen Microsoft's "
                "position in the AI market and enhance their cloud computing services."},
    {"title": "Chelsea Wins Dramatic Match",
     "content": "Chelsea secured a dramatic 3-2 victory in the final minutes of the match, with the "
                "winning goal coming from a well-placed header in stoppage time."},
]

for article in test_articles:
    result = system.analyze(article["title"], article["content"])
    print(f"\n=== {article['title']} ===")
    print(result["enhancement"])


=== Microsoft Acquires AI Startup for $2 Billion ===
This article was classified as **business** (confidence: 36%). Its overall tone is **positive**. Key people mentioned: Satya Nadella. Organizations involved: Microsoft Corporation, Microsoft. Locations referenced: AI.

=== Chelsea Wins Dramatic Match ===
This article was classified as **sport** (confidence: 90%). Its overall tone is **positive**. Organizations involved: Chelsea.


## Conversational interface demo

In [6]:
for q in ["Show me positive tech news", "how many sport articles are there?"]:
    response = system.ask(q)
    print(f"USER: {q}")
    print(f"BOT:  {response['message']}\n")

USER: Show me positive tech news
BOT:  I found 33 articles matching your filter. Here are the top results. (category = tech; sentiment = positive)

USER: how many sport articles are there?
BOT:  Here are 40 results for your query. (category = sport; sentiment = positive)



## Final business insight report

In [7]:
report = system.business_report()
print("=== NewsBot 2.0 - Business Insight Report ===\n")
for insight in report["insights"]:
    print("-", insight)
report["summary"]

=== NewsBot 2.0 - Business Insight Report ===

- 'tech' is the most represented category, accounting for 40 of 200 articles.
- 'entertainment' coverage skews most positive (avg sentiment 0.82), while 'politics' skews most negative (avg sentiment 0.25).
- The classification model achieves a weighted F1-score of 1.00, suitable for automated content routing with human review on low-confidence cases.


{'n_articles': 200,
 'category_counts': {'tech': 40,
  'politics': 40,
  'sport': 40,
  'entertainment': 40,
  'business': 40},
 'sentiment_by_category': {'business': 0.58517,
  'entertainment': 0.8172325,
  'politics': 0.24709749999999997,
  'sport': 0.7952575,
  'tech': 0.6367875},
 'overall_sentiment_avg': 0.6163,
 'best_classifier': 'Naive Bayes',
 'classifier_f1': 1.0,
 'classifier_accuracy': 1.0}

## Project Summary

### Module Integration Checklist
- [x] **Module A - Advanced Content Analysis:** enhanced multi-model classification with confidence scoring, LDA/NMF topic discovery, sentiment evolution, entity relationship mapping
- [x] **Module B - Language Understanding & Generation:** TextRank summarization, TF-IDF semantic search + query expansion, templated insight generation
- [x] **Module C - Multilingual Intelligence:** offline language detection, translation integration (online + offline-demo fallback), cross-lingual sentiment/coverage comparison
- [x] **Module D - Conversational Interface:** rule-based intent classification, filtered + semantic query processing with follow-up context, natural language response generation

### Next Steps for Production
- Swap the offline sample dataset for the full Kaggle BBC dataset (or a live news API) in `data/raw/`.
- Add a scheduled ingestion job (e.g. Celery beat) to keep the Django dashboard's analyzed articles current.
- Upgrade `Summarizer`/`InsightGenerator` to transformer-based backends where internet access to the HuggingFace hub is available, for more fluent abstractive summaries.
- Add authentication + rate limiting to the `/api/analyze/` and `/api/query/` endpoints before any public deployment.